In [2]:
!pip install dbrepo python-dotenv

In [37]:
from dbrepo.RestClient import RestClient
from dotenv import load_dotenv
import os

load_dotenv()
client = RestClient(
    "https://test.dbrepo.tuwien.ac.at/",
    username=os.getenv("DBREPO_USER"),
    password=os.getenv("DBREPO_PASS")
)

DATABASE_ID = os.getenv("DB_ID") #'9fa181a9-de7c-4d44-b367-517a51f31351'  #os.getenv("DB_ID") #"cf27a11d-58e5-4693-856c-e8f3527e3394"

tables = client.get_tables(DATABASE_ID)
for t in tables:
    print(f"Table: {t.name}, ID: {t.id}")
    table_detail = client.get_table(DATABASE_ID, t.id)
    for col in table_detail.columns:
        print(f"  Column: {col.name}, ID: {col.id}")

Table: gdp_data, ID: a7926c82-377b-44f7-819a-b055dff0b5e4
  Column: nuts_code, ID: 868c60ac-ccf4-4c81-944b-fa20ca119833
  Column: ref_year, ID: 3d41938b-8e73-45b2-8c49-e7107916b63d
  Column: gdp, ID: 2c3182d2-7769-49a3-9aa5-fde1b9afb3d1
  Column: currency, ID: 9a2fb0bc-593c-422c-8e6b-e10e6ce9e512
Table: wastewater_data, ID: ffb118dc-d4ff-4c86-b79f-49ae73879ea6
  Column: city_name, ID: e312c51f-a71c-4107-a3dd-79a50460c88c
  Column: ref_year, ID: 5fca89b4-03ed-43cd-b31a-7103587142cd
  Column: metabolite_name, ID: ed0c2260-55d8-48ec-8155-ba5b7f4e8471
  Column: daily_mean_concentration, ID: 84a2570e-d7be-4a06-8af4-a12db02afe95
Table: city_map, ID: e3a73373-d42a-43a3-8c7b-d061910df79f
  Column: nuts_code, ID: c597a750-c2c3-4526-8279-0c74ba77f819
  Column: city_name, ID: 9ff66509-bcc8-48fb-ba00-d054e74e9001


In [38]:
from dbrepo.api.dto import UpdateColumn

mappings = [
    {
        "table_name": "city_map",
        "column": "nuts_code",
        "concept_uri": "http://purl.org/linked-data/sdmx/2009/dimension#refArea"
    },
    {
        "table_name": "city_map",
        "column": "city_name",
        "concept_uri": "http://purl.obolibrary.org/obo/NCIT_C95378"
    },
    {
        "table_name": "gdp_data",
        "column": "ref_year",
        "concept_uri": "http://rs.tdwg.org/dwc/terms/year"        
    },
    {
        "table_name": "gdp_data",
        "column": "currency",
        "concept_uri": "http://purl.org/linked-data/sdmx/2009/attribute#currency",
        "unit_uri": "https://www.omg.org/spec/Commons/QuantitiesAndUnits/hasUnit"
    },
    {
        "table_name": "gdp_data",
        "column": "gdp",
        "concept_uri": "http://purl.org/linked-data/sdmx/2009/measure#obsValue",
        "unit_uri": "https://www.omg.org/spec/Commons/QuantitiesAndUnits/QuantityValue"
    },
    {
        "table_name": "wastewater_data",
        "column": "metabolite_name",
        "concept_uri": "http://purl.obolibrary.org/obo/CHEBI_23367"        
    },
    {
        "table_name": "wastewater_data",
        "column": "daily_mean_concentration",
        "concept_uri": "http://purl.allotrope.org/ontologies/process#AFP_0002800",
        "unit_uri" : "https://www.omg.org/spec/Commons/QuantitiesAndUnits/DerivedUnit"
    }
]

BASE_URL = "https://test.dbrepo.tuwien.ac.at"

for m in mappings:

    print("\n----------------------")
    print(f"{m['table_name']}.{m['column']}")

    table = next((t for t in tables if t.name == m["table_name"]), None)

    if not table:
        print("Table not found")
        continue

    table_detail = client.get_table(DATABASE_ID, table.id)

    col = next((c for c in table_detail.columns if c.name == m["column"]), None)

    if not col:
        print("Column not found")
        continue

    url = f'/api/v1/database/{DATABASE_ID}/table/{table.id}/column/{col.id}'

    response = client._wrapper(method="put", url=url, force_auth=True,
                                 payload=UpdateColumn(concept_uri=m["concept_uri"],
                                                      unit_uri=m.get("unit_uri", "None")))

    print("STATUS:", response.status_code)

    if response.ok:
        print("Success")
    else:
        print("Failed")
        print(response.text)


----------------------
city_map.nuts_code
STATUS: 202
Success

----------------------
city_map.city_name
STATUS: 202
Success

----------------------
gdp_data.ref_year
STATUS: 202
Success

----------------------
gdp_data.currency
STATUS: 202
Success

----------------------
gdp_data.gdp
STATUS: 202
Success

----------------------
wastewater_data.metabolite_name
STATUS: 202
Success

----------------------
wastewater_data.daily_mean_concentration
STATUS: 202
Success


## Brief explanation of used ontologies
The semantic mappings were implemented using statistical and biomedical ontologies, primarily SDMX, ChEBI, and NCIt. SDMX was selected for statistical and regional indicators because it is widely used by organizations such as Eurostat and OECD, while ChEBI and Allotrope ontologies were used for domain-specific chemical and analytical concepts related to wastewater epidemiology data.

## Checking Fields in Database

In [34]:
for t in tables:
    if t.name == "gdp_data":
        tab_id = t.id

In [35]:
tab = client.get_table(database_id = DATABASE_ID, table_id = tab_id)

In [36]:
dict(dict(tab)["columns"][3])

{'id': '9a2fb0bc-593c-422c-8e6b-e10e6ce9e512',
 'name': 'currency',
 'database_id': 'f781c37d-1464-4d86-9791-dc478f8681f2',
 'table_id': 'a7926c82-377b-44f7-819a-b055dff0b5e4',
 'ord': 3,
 'internal_name': 'currency',
 'is_null_allowed': True,
 'type': <ColumnType.VARCHAR: 'varchar'>,
 'alias': None,
 'description': 'Currency pertaining to the Gross Domestic Product (in gdp column)',
 'size': 10,
 'd': None,
 'mean': None,
 'median': None,
 'concept': None,
 'unit': None,
 'enums': [],
 'sets': [],
 'index_length': None,
 'length': None,
 'data_length': None,
 'max_data_length': None,
 'num_rows': None,
 'val_min': None,
 'val_max': None,
 'std_dev': None}